# Asteroid Cost Atlas — Data Explorer

Interactive notebook for querying the latest pipeline output.
Run all cells to see the current state of the atlas, or modify the SQL queries to explore specific questions.

**Prerequisite:** Run `make pipeline` (or at least through `make score-physical`) before using this notebook.

In [ ]:
from pathlib import Path
from asteroid_cost_atlas.utils.query import CostAtlasDB

# Auto-select the latest final-stage parquet
processed = Path("../data/processed")
for pattern in ("sbdb_physical_*.parquet", "sbdb_orbital_*.parquet", "sbdb_enriched_*.parquet"):
    candidates = sorted(processed.glob(pattern))
    if candidates:
        break

if not candidates:
    raise FileNotFoundError("No processed parquet found. Run 'make pipeline' first.")

db = CostAtlasDB(candidates[-1])
print(f"Loaded: {candidates[-1].name}")
print(f"Columns: {db.sql('SELECT * FROM atlas LIMIT 0').columns.tolist()}")

## 1. Catalog Overview

In [ ]:
db.stats()

In [ ]:
db.sql("""
    SELECT
        COUNT(*) AS total,
        COUNT(diameter_km) AS measured_diameter,
        COUNT(diameter_estimated_km) AS estimated_diameter,
        COUNT(rotation_hours) AS has_rotation,
        COUNT(albedo) AS has_albedo,
        COUNT(surface_gravity_m_s2) AS has_gravity,
        COUNT(taxonomy) AS has_taxonomy,
        COUNTIF(neo = 'Y') AS neo_count,
        COUNTIF(pha = 'Y') AS pha_count
    FROM atlas
""")

## 2. Most Accessible Asteroids (lowest delta-v)

In [ ]:
db.top_accessible(n=20)[
    ["name", "delta_v_km_s", "tisserand_jupiter", "inclination_deg",
     "diameter_estimated_km", "diameter_source", "neo", "orbit_class"]
]

## 3. NEA Mining Candidates (T_J in [2, 3), lowest delta-v)

In [ ]:
db.nea_candidates(n=20)[
    ["name", "delta_v_km_s", "tisserand_jupiter", "moid_au",
     "diameter_estimated_km", "diameter_source", "surface_gravity_m_s2"]
]

## 4. Large NEOs with Low Delta-v

The most economically interesting targets: near-Earth, large enough to mine, and cheap to reach.

In [ ]:
db.sql("""
    SELECT name, delta_v_km_s, diameter_estimated_km, diameter_source,
           surface_gravity_m_s2, moid_au, orbit_class, albedo, taxonomy
    FROM atlas
    WHERE neo = 'Y'
      AND diameter_estimated_km > 0.5
      AND delta_v_km_s < 7.0
    ORDER BY delta_v_km_s
    LIMIT 30
""")

## 5. Orbit Class Distribution

In [ ]:
db.sql("""
    SELECT orbit_class,
           COUNT(*) AS count,
           ROUND(AVG(delta_v_km_s), 2) AS avg_delta_v,
           ROUND(AVG(diameter_estimated_km), 2) AS avg_diameter_km,
           ROUND(AVG(inclination_deg), 2) AS avg_inclination
    FROM atlas
    WHERE orbit_class IS NOT NULL
    GROUP BY orbit_class
    ORDER BY count DESC
    LIMIT 15
""")

## 6. Delta-v Distribution

In [ ]:
hist = db.delta_v_histogram(bin_width=1.0)
hist

## 7. Diameter Source Breakdown

In [ ]:
db.sql("""
    SELECT diameter_source,
           COUNT(*) AS count,
           ROUND(MIN(diameter_estimated_km), 4) AS min_km,
           ROUND(MEDIAN(diameter_estimated_km), 4) AS median_km,
           ROUND(MAX(diameter_estimated_km), 2) AS max_km
    FROM atlas
    WHERE diameter_source IS NOT NULL
    GROUP BY diameter_source
    ORDER BY count DESC
""")

## 8. Physical Feasibility: Best Candidates

Asteroids with all three physical scores (gravity, rotation feasibility, regolith likelihood).

In [ ]:
db.sql("""
    SELECT name, delta_v_km_s, diameter_estimated_km,
           ROUND(surface_gravity_m_s2, 6) AS gravity_m_s2,
           ROUND(rotation_feasibility, 3) AS rot_feasibility,
           ROUND(regolith_likelihood, 3) AS regolith,
           rotation_hours, neo, orbit_class
    FROM atlas
    WHERE rotation_feasibility IS NOT NULL
      AND regolith_likelihood > 0.5
      AND rotation_feasibility > 0.5
    ORDER BY delta_v_km_s
    LIMIT 20
""")

## 9. Taxonomy Distribution (from LCDB)

In [ ]:
db.sql("""
    SELECT taxonomy,
           COUNT(*) AS count,
           ROUND(AVG(albedo), 3) AS avg_albedo,
           ROUND(AVG(diameter_estimated_km), 2) AS avg_diameter_km,
           ROUND(AVG(delta_v_km_s), 2) AS avg_delta_v
    FROM atlas
    WHERE taxonomy IS NOT NULL
    GROUP BY taxonomy
    ORDER BY count DESC
    LIMIT 15
""")

## 10. Custom Query

Edit the SQL below to explore anything. The view is called `atlas`.

In [ ]:
db.sql("""
    SELECT *
    FROM atlas
    LIMIT 10
""")

In [ ]:
db.close()